# Data Collection: Claude (Anthropic API)

This notebook queries Claude via the Anthropic API for each question in the question bank and saves responses to `data/responses/claude.jsonl`.

**Before running:** complete the setup in [Environment Setup](setup.md) and ensure `ANTHROPIC_API_KEY` is set.

In [ ]:
import anthropic
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

## Configuration

In [ ]:
MODEL = "claude-opus-4-6"          # or claude-sonnet-4-6 for faster/cheaper runs
TEMPERATURE = 0.0                   # deterministic for replication
MAX_TOKENS = 1024
OUTPUT_FILE = Path("data/responses/claude.jsonl")
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

## Load Question Bank

In [ ]:
import pandas as pd

# Load questions from CSV (generated from figures/question_bank.csv)
questions = pd.read_csv("../figures/question_bank.csv")
print(f"Loaded {len(questions)} questions")
questions.head()

## System Prompt

A neutral system prompt is used across all models to minimise prompt-induced variance.

In [ ]:
SYSTEM_PROMPT = """You are a knowledgeable assistant answering questions about early church history.
Provide accurate, detailed, and nuanced responses based on historical scholarship.
Acknowledge uncertainty or scholarly debate where it exists.
Do not refuse historically answerable questions on the grounds of contemporary theological controversy."""

## Run Collection

In [ ]:
def query_claude(question_text: str) -> dict:
    """Query Claude and return the response with metadata."""
    message = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": question_text}],
    )
    return {
        "response": message.content[0].text,
        "tokens_used": message.usage.input_tokens + message.usage.output_tokens,
        "model_version": message.model,
    }


results = []
with open(OUTPUT_FILE, "w") as f:
    for _, row in questions.iterrows():
        print(f"Querying {row['question_id']}...", end=" ")
        try:
            result = query_claude(row["question"])
            record = {
                "question_id": row["question_id"],
                "figure": row["figure"],
                "model": MODEL,
                "model_version": result["model_version"],
                "prompt": row["question"],
                "response": result["response"],
                "temperature": TEMPERATURE,
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "tokens_used": result["tokens_used"],
            }
            f.write(json.dumps(record) + "\n")
            results.append(record)
            print(f"done ({result['tokens_used']} tokens)")
        except Exception as e:
            print(f"ERROR: {e}")
        time.sleep(0.5)  # gentle rate limiting

print(f"\nSaved {len(results)} responses to {OUTPUT_FILE}")

## Inspect Results

In [ ]:
# Preview first response
if results:
    r = results[0]
    print(f"Q: {r['prompt']}\n")
    print(f"A: {r['response']}\n")
    print(f"Tokens: {r['tokens_used']} | Model: {r['model_version']}")